# Tarea: Geometría de Datos en Producción
## Escalamiento y Discretización (Google Play Store)

**Objetivo:**
En la clase anterior demostramos cómo las variables con escalas gigantescas secuestran el espacio vectorial y destruyen los algoritmos de Machine Learning. Ahora es su turno de aplicar estos conceptos arquitectónicos al *dataset* de la Google Play Store que limpiaron la semana pasada.

**Instrucciones:**
Resuelvan los siguientes 4 retos programando directamente en las celdas de código de este Notebook. Para cada reto, respondan a la pregunta analítica utilizando un comentario o una celda Markdown.

In [1]:
import pandas as pd
import numpy as np

# Instrucción: Carga tu archivo limpio de la semana 4 ('playstore_limpio.csv').
# Si por alguna razón no lo tienes, usa el siguiente bloque para descargar y pre-limpiar lo básico:

url = "../datasets/playstore_limpio.csv"
df = pd.read_csv(url)

# Limpieza rápida (Mockup del resultado de la Semana 4)
# df.drop_duplicates(subset=['App'], inplace=True)
# df = df[df['Installs'] != 'Free'] # Quitar la fila de error ("Life Made WI-Fi")
# df['Installs'] = df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False).astype(float)
# df['Reviews'] = df['Reviews'].astype(float)
# df['Rating'] = df['Rating'].astype(float)
# # Rellenar ratings nulos con la mediana para que el escalamiento no falle
# df['Rating'].fillna(df['Rating'].median(), inplace=True)

print("Datos listos. Total de apps:", len(df))

Datos listos. Total de apps: 9636


---
### Reto 1: La Trampa de Min-Max en un Mundo Viral
La columna `Installs` (Descargas) representa un reto enorme porque vivimos en un mundo asimétrico: hay millones de apps con 10 descargas y unas cuantas con 1,000 millones (como WhatsApp).

1. Crea una nueva columna llamada `Installs_MinMax` aplicando la fórmula $(X - Min) / (Max - Min)$.
2. Ejecuta un `.describe()` sobre tu nueva columna.
3. **Pregunta de Análisis:** Observa el percentil 75% (`75%`) de tu nueva columna comprimida. ¿Qué valor numérico tiene? Como arquitecto de datos, explica qué nos dice este número sobre el riesgo de usar *Min-Max Scaler* en variables virales (con *outliers* masivos).

In [2]:

print(df['Installs'].describe())


# 1. Aplica la fórmula Min-Max a la columna 'Installs'
df['Installs_MinMax'] = (df["Installs"]-df["Installs"].min())/(df["Installs"].max()-df["Installs"].min())

# 2. Muestra el resumen estadístico
print(df['Installs_MinMax'].describe())



count    9.636000e+03
mean     7.796048e+06
std      5.382106e+07
min      0.000000e+00
25%      1.000000e+03
50%      1.000000e+05
75%      1.000000e+06
max      1.000000e+09
Name: Installs, dtype: float64
count    9636.000000
mean        0.007796
std         0.053821
min         0.000000
25%         0.000001
50%         0.000100
75%         0.001000
max         1.000000
Name: Installs_MinMax, dtype: float64


### RESPUESTA A LA PREGUNTA (Escribe aquí tu análisis):
75% = 0.001

no se distribuyen o representan muy bien que digamos los cuartiles cuando hay outliers masivos, sería bueno usar una escala logarítmica para representar mejor la cantidad de descargas por cada rango de apps.


---
### Reto 2: Estandarización de Calificaciones (Z-Score)
La columna `Rating` va de 1.0 a 5.0. Vamos a estandarizarla para que su media sea 0 y su desviación estándar sea 1 (Campana de Gauss). En una distribución normal, cualquier valor con un $Z-Score$ menor a $-3$ es estadísticamente una "anomalía negativa" extrema.

1. Crea la columna `Rating_Z` aplicando la fórmula $(X - \mu) / \sigma$.
2. Escribe una máscara booleana para filtrar el *DataFrame* y encontrar cuántas aplicaciones tienen un `Rating_Z` menor a `-3.0`.
3. **Pregunta de Análisis:** ¿Cuántas aplicaciones son "anomalías negativas" extremas? ¿Qué calificación original (en estrellas) tenían esas apps para merecer ese Z-Score?

In [3]:
# 1. Aplica la fórmula Z-Score a la columna 'Rating'
df['Rating_Z'] = (df["Rating"]-df["Rating"].mean())/df["Rating"].std()

# 2. Filtra las apps con Z < -3.0
anomalias_negativas = df[ df['Rating_Z'] < -3.0 ]
print("Total de anomalías negativas:", len(anomalias_negativas))
print(anomalias_negativas[['App', 'Rating', 'Rating_Z']].head())


Total de anomalías negativas: 147
                                                 App  Rating  Rating_Z
448                    Just She - Top Lesbian Dating     1.9 -4.236257
450   EliteSingles – Dating for Single Professionals     2.5 -3.118390
457                           Sugar Daddy Dating App     2.5 -3.118390
524                          House party - live chat     1.0 -5.913058
1761                                        VeinSeek     2.5 -3.118390



### RESPUESTA A LA PREGUNTA:
147 anomalias negaticas encontradas. tienen un rating menor o igual a 2.5 estrelas

---
### Reto 3: Categorización Comercial (El Cuchillo Lógico `pd.cut`)
El equipo de marketing no entiende de estrellas decimales (ej. 4.3). Quieren que las aplicaciones se clasifiquen en 4 etiquetas simples para el *frontend* de la tienda.

1. Utiliza `pd.cut()` sobre la columna original `Rating` para crear una nueva columna llamada `Categoria_Calidad`.
2. Utiliza los siguientes límites (`bins`): `0.0, 3.0, 4.0, 4.5, 5.0`.
3. Asigna las siguientes etiquetas (`labels`): `'Pésima', 'Regular', 'Buena', 'Excelente'`.
4. Ejecuta un `.value_counts()` para ver cuántas hay de cada una. ¿Quedaron desbalanceadas?

In [4]:
# 1 y 2. Aplica pd.cut con los bins y labels especificados
df['Categoria_Calidad'] = pd.cut(df["Rating"],bins=[0.0,3.0,4.0,4.5,5.0],labels=['Pésima', 'Regular', 'Buena', 'Excelente'])

# 4. Muestra los conteos
print(df['Categoria_Calidad'].value_counts())



Categoria_Calidad
Buena        4066
Regular      2054
Excelente    1701
Pésima        359
Name: count, dtype: int64


### RESPUESTA:
Si quedaron desbalanceadas

---
### Reto 4: Cuartiles de *Engagement* (El Cuchillo Estadístico `pd.qcut`)
A diferencia del equipo de marketing, tu modelo de Machine Learning necesita grupos balanceados para no desarrollar sesgos. Vamos a clasificar las aplicaciones basándonos en su volumen de reseñas (`Reviews`).

1. Utiliza `pd.qcut()` sobre la columna `Reviews` para crear 4 grupos de igual tamaño (`q=4`). Llama a la nueva columna `Nivel_Engagement`.
2. Asigna las etiquetas: `'Bajo', 'Medio', 'Alto', 'Viral'`.
3. Ejecuta un `.value_counts()` sobre `Nivel_Engagement`.
4. **Pregunta de Análisis:** Compara este conteo con el que obtuviste en el Reto 3. ¿Por qué el algoritmo de clasificación preferiría predecir la columna `Nivel_Engagement` en lugar de la columna `Categoria_Calidad`?

In [5]:
# 1 y 2. Aplica pd.qcut para generar 4 categorías de igual tamaño
df['Nivel_Engagement'] = pd.qcut(df["Reviews"],q=4,labels=["Bajo","Medio","Alto","Viral"])

# 3. Muestra los conteos (Deberían ser casi idénticos en cada categoría)
print(df['Nivel_Engagement'].value_counts())



Nivel_Engagement
Bajo     2424
Viral    2409
Alto     2407
Medio    2396
Name: count, dtype: int64


#### RESPUESTA A LA PREGUNTA:
Por que Nivel_Engagement esta balanceada.